# Gold Layer

This notebook creates the final analysis output from the cleaned Silver dataset.

In [0]:
#Import Libraries

import pyspark.sql.functions as F
from pyspark.sql.types import *




In [0]:
#Define Catalog Name
catalog_name = "carnot-catalog"

In [0]:
#Read Silver Table

parcel_timeseries_silver = spark.table(
    f"`{catalog_name}`.silver.silver_parcel_timeseries"
)

display(parcel_timeseries_silver.limit(5))

parcel_id,reading_date,ndvi_value,temperature_c,rainfall_mm,sensor_status,mill_id,crop_type,sowing_date,area_hectares
PARCEL_015,2026-01-11,0.257,34.0,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_014,2026-03-18,0.769,18.0,7.8,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_014,2026-01-07,0.288,19.4,0.0,ok,MILL_NORTH,sugarcane,2026-01-05,9.39
PARCEL_015,2026-02-13,0.802,16.3,0.0,ok,MILL_SOUTH,sugarcane,2026-01-06,4.87
PARCEL_019,2026-01-30,0.335,24.7,0.0,ok,MILL_SOUTH,sugarcane,2026-01-18,10.19


Ignore rows where sensor_status is bad

In [0]:
#shows unique sensor status values

parcel_timeseries_silver.select("sensor_status").distinct().display()

sensor_status
ok
na
error
nan
null


In [0]:
#Ignore rows where sensor_status is bad

gold_df = parcel_timeseries_silver.filter(
    F.col("sensor_status") == "ok"
)

In [0]:
#display(gold_df)

In [0]:
## Calculate difference between reading date and sowing date


gold_df = gold_df.withColumn(
    "days_from_sowing",
    F.datediff(
        F.col("reading_date"),
        F.col("sowing_date")
    )
)

In [0]:
#display(gold_df)

In [0]:
# # Readings from 30 days before sowing


before_df = gold_df.filter(
    (F.col("days_from_sowing") >= -30) &
    (F.col("days_from_sowing") < 0)
)

In [0]:
#display(before_df)

In [0]:
# Readings from 30 days after sowing

after_df = gold_df.filter(
    (F.col("days_from_sowing") > 0) &
    (F.col("days_from_sowing") <= 30)
)

In [0]:
#display(after_df)

In [0]:
# Mean ndvi before sowing by crop type
before_agg = before_df.groupBy("crop_type").agg(
    F.avg("ndvi_value").alias("mean_ndvi_before")
)

In [0]:
#display(before_agg)

In [0]:
# Mean ndvi after sowing and parcel counts
after_agg = after_df.groupBy("crop_type").agg(
    F.avg("ndvi_value").alias("mean_ndvi_after"),
    F.countDistinct("parcel_id").alias("n_parcels")

)

In [0]:
#display(after_agg)

In [0]:
# Final analytical output
result_df = before_agg.join(
    after_agg,
    "crop_type",
    "inner"
)

In [0]:
# formatting result df

result_df = result_df.select(
    "crop_type",
    F.round("mean_ndvi_before", 3).alias("mean_ndvi_before"),
    F.round("mean_ndvi_after", 3).alias("mean_ndvi_after"),
    "n_parcels"
)


display(result_df.limit(5))

crop_type,mean_ndvi_before,mean_ndvi_after,n_parcels
sugarcane,0.177,0.336,19
soybean,0.171,0.313,4
wheat,0.176,0.31,2


In [0]:
#Save analytical output


result_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog_name}`.gold.gold_crop_ndvi_analysis"
    )


In [0]:
# save output in csv

result_df.write.mode("overwrite").option("header", True).csv(f"/Volumes/{catalog_name}/gold/csv_exports/gold_crop_ndvi_analysis_csv")